In [196]:
import numpy as np
from typing import List
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import copy
from qiskit.quantum_info import random_unitary
import matplotlib as mpl
import time
from line_profiler import LineProfiler
from pathlib import Path
import gzip
import pickle

In [17]:
def cost_function1(n: int) -> int:
    return (4 - 3*2**(3-n))

In [18]:
def rz_gate(theta: float):
    return np.array([[np.exp(-1j * theta/2), 0], [0, np.exp(1j * theta/2)]])

In [19]:
# Creating the Tau gate sets
def odd_k(denom: int) -> List[int]:
    k_list = []
    highest = denom//2 - 1
    k_list = list(range(-highest, highest+1, 2))
    return k_list
    
def tau_set(n: int):
    tau = []
    denom = 2**(n-1)
    k_list = odd_k(denom)
    cost = cost_function1(n)
    for k in k_list:
        gate = rz_gate(k*np.pi/denom)
        tau.append({"gate": gate, "t_count": cost, "name": f"Rz({k}*π/{denom})", "magic_gate": True})
    return tau

In [20]:
# Creating the logical base gate set number 1
set1 = []

pauli_x = np.array([[0, 1], [1, 0]])
set1.append({"gate": pauli_x, "t_count": 0, "name": "X", "magic_gate": False})

pauli_y = np.array([[0, -1j], [1j, 0]])
set1.append({"gate": pauli_y, "t_count": 0, "name": "Y", "magic_gate": False})

pauli_z = np.array([[1, 0], [0, -1]])
set1.append({"gate": pauli_z, "t_count": 0, "name": "Z", "magic_gate": False})

hadamard = 1/np.sqrt(2) * np.array([[1, 1], [1, -1]])
set1.append({"gate": hadamard, "t_count": 0, "name": "H", "magic_gate": False})

s_gate = rz_gate(np.pi/2)
set1.append({"gate": s_gate, "t_count": 0, "name": "S", "magic_gate": False})

sdg_gate = rz_gate(-np.pi/2)
set1.append({"gate": sdg_gate, "t_count": 0, "name": "Sdg", "magic_gate": False})

set1 = set1 + tau_set(3)
print(set1)

[{'gate': array([[0, 1],
       [1, 0]]), 't_count': 0, 'name': 'X', 'magic_gate': False}, {'gate': array([[ 0.+0.j, -0.-1.j],
       [ 0.+1.j,  0.+0.j]]), 't_count': 0, 'name': 'Y', 'magic_gate': False}, {'gate': array([[ 1,  0],
       [ 0, -1]]), 't_count': 0, 'name': 'Z', 'magic_gate': False}, {'gate': array([[ 0.70710678,  0.70710678],
       [ 0.70710678, -0.70710678]]), 't_count': 0, 'name': 'H', 'magic_gate': False}, {'gate': array([[0.70710678-0.70710678j, 0.        +0.j        ],
       [0.        +0.j        , 0.70710678+0.70710678j]]), 't_count': 0, 'name': 'S', 'magic_gate': False}, {'gate': array([[0.70710678+0.70710678j, 0.        +0.j        ],
       [0.        +0.j        , 0.70710678-0.70710678j]]), 't_count': 0, 'name': 'Sdg', 'magic_gate': False}, {'gate': array([[0.92387953+0.38268343j, 0.        +0.j        ],
       [0.        +0.j        , 0.92387953-0.38268343j]]), 't_count': 1, 'name': 'Rz(-1*π/4)', 'magic_gate': True}, {'gate': array([[0.92387953-0.38268343j

In [93]:
def remove_global_phase(U, tol=1e-12):
    """
    det = np.linalg.det(U)
    if abs(det) < tol or det == 0:
        return U
    else:
        return U / np.exp(1j*np.angle(det)/2)
    """
    det = U[0,0]*U[1,1] - U[0,1]*U[1,0]
    return U / np.sqrt(det + 0j)

In [22]:
# Adding exotic magic state to set1
theta = np.arctan(np.sqrt((np.sqrt(5) - 1) / 2))
exotic_magic_state = np.array([[np.cos(theta/2), -np.sin(theta/2)], [np.sin(theta/2), np.cos(theta/2)]])
exotic_entry = {"gate": exotic_magic_state, "t_count": 0, "name": "EMS", "magic_gate": True}

exotic_set1 = set1.copy()
exotic_set1.append(exotic_entry)

In [23]:
gate_dict = {}
gate_dict['I'] = np.eye(2, dtype='complex')
gate_dict['X'] = np.array([[0, 1], [1, 0]])
gate_dict['Y'] = np.array([[0, -1j], [1j, 0]])
gate_dict['Z'] = np.array([[1, 0], [0, -1]])
gate_dict['H'] = 1/np.sqrt(2) * np.array([[1, 1], [1, -1]])
gate_dict['S'] = rz_gate(np.pi/2)
gate_dict['Sdg'] = rz_gate(-np.pi/2)
gate_dict['Rz(-1*π/4)'] = np.array([[0.92387953+0.38268343j, 0.        +0.j        ],
       [0.        +0.j        , 0.92387953-0.38268343j]])
gate_dict['Rz(1*π/4)'] = np.array([[0.92387953-0.38268343j, 0.        +0.j        ],
       [0.        +0.j        , 0.92387953+0.38268343j]])
gate_dict['EMS'] = np.array([[np.cos(theta/2), -np.sin(theta/2)], [np.sin(theta/2), np.cos(theta/2)]])

In [29]:
def Lie_generator(U, eps: float = 1e-12):
    if (np.array_equal(np.eye(2, dtype='complex'), U)):
        return (0, 0, 0)
    
    U = remove_global_phase(U)
    """
    theta = 2*np.arccos(np.trace(U)/2)
    
    denom = np.sin(theta/2.0)
    if (abs(denom) < eps):
        return (0, 0, 0)
    
    n1 = 1j * np.trace(pauli_x @ U) / (2*np.sin(theta/2))
    n2 = 1j * np.trace(pauli_y @ U) / (2*np.sin(theta/2))
    n3 = 1j * np.trace(pauli_z @ U) / (2*np.sin(theta/2))
    
    alpha = np.round(np.real(theta/2 * n1), 8)
    beta = np.round(np.real(theta/2 * n2), 8)
    gamma = np.round(np.real(theta/2 * n3), 8)
    """
    V = remove_global_phase(U)
    A = 1j * logm(V)
    A = 0.5 * (A + A.conj().T)
    alpha = 0.5 * np.trace(A @ pauli_x).real
    beta  = 0.5 * np.trace(A @ pauli_y).real
    gamma = 0.5 * np.trace(A @ pauli_z).real
    
    return (alpha, beta, gamma)

In [ ]:
def key_from_U(U: np.ndarray, grid: float = 1e-7):
    # U = aI + i(xX + yY + zZ)
    det = U[0,0]*U[1,1] - U[0,1]*U[1,0]
    U = U / np.sqrt(det + 0j)
    inv_grid = 1.0 / grid
    components = np.array([
         0.5 * (U[0,0] + U[1,1]),
        -0.5j * (U[0,1] + U[1,0]),
         0.5 * (U[0,1] - U[1,0]),
        -0.5j * (U[0,0] - U[1,1])
    ])
    
    return tuple(np.round(components * inv_grid))

In [207]:
class SequenceDB:
    def __init__(self):
        self.vecs = set() # (a,x,y,z)
        self.seqs = [] # sequence of gates as a list of list of strings
        self.magic_count = [] # total magic count
        self.t_count = [] # total t count
        self.unitaries = [] # 2x2 unitaries
        
    def add(self, v, seq, magic_count, t_count, matrix):
        self.vecs.add(v)
        self.seqs.append([str(g) for g in seq])
        self.magic_count.append(float(magic_count))
        self.t_count.append(float(t_count))
        self.unitaries.append(np.array(matrix))
        
    def save(self, filepath):
        with open(filepath, 'wb') as f:
            pickle.dump(self, f)

    @classmethod
    def load(cls, filepath):
        with open(filepath, 'rb') as f:
            return pickle.load(f)

In [64]:
def generate_sequences(base_gate_set, max_magic_count, previous=None):
    clifford_gates = [g for g in base_gate_set if not g['magic_gate']]
    magic_gates = [g for g in base_gate_set if g['magic_gate']]
    
    if previous is None:
        db = SequenceDB()
        db.add((0.0, 0.0, 0.0), ['I'], 0.0, 0.0, np.eye(2, dtype='complex'))
        for g in clifford_gates:
            db.add(key_from_U(g['gate']), [g['name']], 0.0, 0.0, g['gate'])
        for g in magic_gates:
            db.add(key_from_U(g['gate']), [g['name']], 1.0, 1.0, g['gate'])
    else:
        db = copy.deepcopy(previous)
    
    if max(db.magic_count) >= max_magic_count:
        print("Max Magic Count Reached")
        return

    for i, seq in enumerate(db.seqs):
        # from seq, obtaining the 2x2 matrix
        current_unitary = db.unitaries[i]

        # last gate in sequence is a clifford? 
        clifford = False if (seq[-1] == 'Rz(-1*π/4)' or seq[-1] == 'Rz(1*π/4)' or seq[-1] == 'EMS') else True
        
        # looping over all possible gates, either only Clifford or only magic
        candidates = magic_gates if clifford else clifford_gates
        for gate_info in candidates:
            # one needs to be a magic gate and the other needs to be a clifford
            if gate_info['magic_gate'] != clifford:
                continue
            
            # if new unitary already exists skip, otherwise add to database
            temp_unitary = gate_info['gate'] @ current_unitary
            temp_vector = key_from_U(temp_unitary)
            if temp_vector in db.vecs:
                continue
            else:
                temp_magic_count = 1 if clifford == True else 0
                temp_t_count = 1 if (gate_info['name'] == 'Rz(-1*π/4)' or gate_info['name'] == 'Rz(1*π/4)') else 0
                # check if the new unitary has exceeded the max number of magic states
                if db.magic_count[i] + temp_magic_count > max_magic_count:
                    continue
                else:
                    temp_seq = seq + [gate_info['name']]
                    db.add(temp_vector, temp_seq, db.magic_count[i] + temp_magic_count, db.t_count[i] + temp_t_count, temp_unitary)
    return db

In [154]:
exotic_set1_max2 = generate_sequences(exotic_set1, 2)

In [155]:
exotic_set1_max3 = generate_sequences(exotic_set1, 3, exotic_set1_max2)

In [156]:
exotic_set1_max4 = generate_sequences(exotic_set1, 4, exotic_set1_max3)

In [118]:
exotic_set1_max5 = generate_sequences(exotic_set1, 5, exotic_set1_max4)

In [119]:
exotic_set1_max6 = generate_sequences(exotic_set1, 6, exotic_set1_max5)

In [122]:
exotic_set1_max7 = generate_sequences(exotic_set1, 7, exotic_set1_max6)

In [157]:
exotic_set1_max8 = generate_sequences(exotic_set1, 8, exotic_set1_max7)

In [ ]:
exotic_set1_max9 = generate_sequences(exotic_set1, 9, exotic_set1_max8)

In [ ]:
exotic_set1_max10 = generate_sequences(exotic_set1, 10, exotic_set1_max9)

In [158]:
print(len(exotic_set1_max8.magic_count))

24617793


In [128]:
print(max(exotic_set1_max7.t_count))

7.0


In [129]:
profiler = LineProfiler()
profiler.add_function(generate_sequences)
profiler.add_function(key_from_U)        # add any functions you suspect
profiler.run('generate_sequences(exotic_set1, 7, exotic_set1_max6)')
profiler.print_stats()

Timer unit: 1e-09 s

Total time: 3167.53 s
File: /var/folders/yh/ktq4n6y1305b9mr9r8_d4znw0000gn/T/ipykernel_18417/4265032017.py
Function: generate_sequences at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def generate_sequences(base_gate_set, max_magic_count, previous=None):
     2         1       6000.0   6000.0      0.0      clifford_gates = [g for g in base_gate_set if not g['magic_gate']]
     3         1       1000.0   1000.0      0.0      magic_gates = [g for g in base_gate_set if g['magic_gate']]
     4                                               
     5         1          0.0      0.0      0.0      if previous is None:
     6                                                   db = SequenceDB()
     7                                                   db.add((0.0, 0.0, 0.0), ['I'], 0.0, 0.0, np.eye(2, dtype='complex'))
     8                                                   for g in clifford_gates:
     

In [208]:
exotic_set1_max8.save("exotic_set1_max8.pkl")

AttributeError: 'SequenceDB' object has no attribute 'save'

In [ ]:
exotic_set1_max8 = SequenceDB.load("exotic_set1_max8.pkl")

Given an error tolerance epsilon, find an approximation within the error tolerance

In [185]:
def distance(U, V):
    # U and V both have the form (a, x, y, z)
    return (2 * np.sqrt(1 - np.dot(U, V))).real

In [186]:
# returns a list of all unitaries within the error tolerance, and the indexes in the database
def approximate_unitary(unitary, error_tolerance, db):
    unitary = key_from_U(unitary)
    answer_list = []
    answer_idxs = []
    for i, u in enumerate(db.unitaries):
        if distance(key_from_U(u), unitary) <= error_tolerance:
            answer_list.append(u)
            answer_idxs.append(i)
    
    if len(answer_idxs) == 0:
        print("Database doesn't contain any unitaries within the error tolerance.")
        return None
    else:
        return answer_list, answer_idxs

In [187]:
alpha1 = -0.274220
rz_alpha1_matrix = np.array([[np.exp(-1j*alpha1/2), 0], [0, np.exp(1j*alpha1/2)]])

unitaries, idxs = approximate_unitary(rz_alpha1_matrix, 0.01, exotic_set1_max8)